## Movie Recommendation Data Preprocessing

### 1. Import Libraries

In [36]:
# Core libraries for data cleaning, numeric ops, and parsing stringified JSON columns
import ast
import numpy as np
import pandas as pd
from pathlib import Path

### 2. Load Datasets

In [37]:
project_path = Path().resolve().parent

In [38]:
# Load raw TMDB movies and credits datasets
movies_df = pd.read_csv(project_path / "Data" / "tmdb_5000_movies.csv")
movie_credits_df = pd.read_csv(project_path / "Data" / "tmdb_5000_credits.csv")

In [39]:
movies_df.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [40]:
movie_credits_df.head(1)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


### 3. Merge Movie Metadata With Credits

In [41]:
# Merge movies metadata with credits data on the shared movie id
movies_with_credits_df = movies_df.merge(movie_credits_df, left_on="id", right_on="movie_id")

In [42]:
movies_with_credits_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   str    
 2   homepage              1712 non-null   str    
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   str    
 5   original_language     4803 non-null   str    
 6   original_title        4803 non-null   str    
 7   overview              4800 non-null   str    
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   str    
 10  production_countries  4803 non-null   str    
 11  release_date          4802 non-null   str    
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   str    
 15  status                4803 non-n

In [43]:
movies_with_credits_df.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,spoken_languages,status,tagline,title_x,vote_average,vote_count,movie_id,title_y,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


### 4. Select Required Features

In [44]:
# Keep only the columns needed to build recommendation tags, and standardize the title column name
selected_movie_columns = [
    "id",
    "genres",
    "keywords",
    "overview",
    "production_companies",
    "title_x",
    "cast",
    "crew",
]

movie_features_df = movies_with_credits_df[selected_movie_columns].rename(columns={"title_x": "title"})

In [45]:
movie_features_df.head(1)

,id,genres,keywords,overview,production_companies,title,cast,crew
0,19995,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","In the 22nd century, a paraplegic Marine is di...","[{""name"": ""Ingenious Film Partners"", ""id"": 289...",Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


### 5. Clean Missing And Duplicate Records

In [46]:
movie_features_df.isna().sum()

id                      0
genres                  0
keywords                0
overview                3
production_companies    0
title                   0
cast                    0
crew                    0
dtype: int64

In [47]:
# Drop rows with missing values (few overview entries are empty)
movie_features_df.dropna(inplace=True)

In [48]:
movie_features_df.isna().sum()

id                      0
genres                  0
keywords                0
overview                0
production_companies    0
title                   0
cast                    0
crew                    0
dtype: int64

In [49]:
movie_features_df.duplicated().sum()

np.int64(0)

### 6. Convert JSON-Like Text Columns

In [50]:
movie_features_df["genres"].iloc[0]

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [51]:
def extract_names(json_text):
    """Parse a JSON-like string column and return a list of 'name' values with spaces removed."""
    names = []

    for item in ast.literal_eval(json_text):
        names.append(item["name"].replace(" ", ""))

    return names


def extract_top_cast_names(json_text):
    """Return the names of the top 3 billed cast members (spaces removed)."""
    cast_names = []
    cast_limit = 3

    for cast_member in ast.literal_eval(json_text):
        cast_names.append(cast_member["name"].replace(" ", ""))

        if len(cast_names) == cast_limit:
            break

    return cast_names


def extract_director_name(json_text):
    """Return the director's name (spaces removed) as a single-item list, or an empty list if not found."""
    for crew_member in ast.literal_eval(json_text):
        if crew_member["job"] == "Director":
            return [crew_member["name"].replace(" ", "")]

    return []


In [52]:
# Convert JSON-like string columns into clean lists of names (no spaces, so multi-word names count as one token)
movie_features_df["genres"] = movie_features_df["genres"].apply(extract_names)
movie_features_df["keywords"] = movie_features_df["keywords"].apply(extract_names)
movie_features_df["production_companies"] = movie_features_df["production_companies"].apply(extract_names)
movie_features_df["cast"] = movie_features_df["cast"].apply(extract_top_cast_names)
movie_features_df["crew"] = movie_features_df["crew"].apply(extract_director_name)

In [53]:
# Split the overview text into individual words so it can be combined with other tag lists
movie_features_df["overview"] = movie_features_df["overview"].apply(lambda overview: overview.split())

### 7. Build Recommendation Tags

In [54]:
# Combine all relevant features into a single 'tags' list per movie
movie_features_df["tags"] = (
    movie_features_df["genres"]
    + movie_features_df["keywords"]
    + movie_features_df["overview"]
    + movie_features_df["production_companies"]
    + movie_features_df["cast"]
    + movie_features_df["crew"]
)

In [55]:
# Drop columns already merged into 'tags' to keep the dataframe lean
movie_features_df.drop(columns=["keywords", "production_companies"], inplace=True)

In [56]:
movie_features_df.head(1)

,id,genres,overview,title,cast,crew,tags
0,19995,"[Action, Adventure, Fantasy, ScienceFiction]","[In, the, 22nd, century,, a, paraplegic, Marin...",Avatar,"[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],"[Action, Adventure, Fantasy, ScienceFiction, c..."


In [57]:
# Join tag words into a single space-separated string per movie
movie_features_df["tags"] = movie_features_df["tags"].apply(lambda tag_words: " ".join(tag_words))

In [58]:
# Lowercase tags for consistent vectorization
movie_features_df["tags"] = movie_features_df["tags"].apply(lambda tags: tags.lower())

### 8. Vectorize Tags And Calculate Similarity

In [59]:
# Convert tags into TF-IDF vectors (ignoring common English stop words, capped vocabulary size)
from sklearn.feature_extraction.text import TfidfVectorizer

tag_vectorizer = TfidfVectorizer(stop_words="english",max_features=7000)
tag_vectors = tag_vectorizer.fit_transform(movie_features_df["tags"]).toarray()


In [60]:
# Import cosine similarity to measure how alike two movies' tag vectors are
from sklearn.metrics.pairwise import cosine_similarity

In [61]:
# Compute pairwise cosine similarity between all movies based on their tag vectors
movie_similarity_matrix = cosine_similarity(tag_vectors, tag_vectors).astype(np.float16)

In [62]:
movie_similarity_matrix

array([[1.      , 0.019   , 0.010216, ..., 0.004723, 0.004856, 0.      ],
       [0.019   , 1.      , 0.01144 , ..., 0.01653 , 0.      , 0.      ],
       [0.010216, 0.01144 , 1.      , ..., 0.01501 , 0.      , 0.      ],
       ...,
       [0.004723, 0.01653 , 0.01501 , ..., 1.      , 0.0256  , 0.03152 ],
       [0.004856, 0.      , 0.      , ..., 0.0256  , 1.      , 0.01502 ],
       [0.      , 0.      , 0.      , ..., 0.03152 , 0.01502 , 1.      ]],
      shape=(4800, 4800), dtype=float16)

In [63]:
movie_features_df.info()

<class 'pandas.DataFrame'>
Index: 4800 entries, 0 to 4802
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        4800 non-null   int64 
 1   genres    4800 non-null   object
 2   overview  4800 non-null   object
 3   title     4800 non-null   str   
 4   cast      4800 non-null   object
 5   crew      4800 non-null   object
 6   tags      4800 non-null   str   
dtypes: int64(1), object(4), str(2)
memory usage: 300.0+ KB


### 9. Dump Pickle File

In [64]:
# Save the processed dataframe and similarity matrix for use in the Streamlit app
import pickle
import gzip

with gzip.open(project_path / "Data" / "movies_data.pkl","wb") as file:
    pickle.dump((movie_features_df, movie_similarity_matrix),file)